# Core Machine Learning — Interview Preparation Notebook

This notebook covers essential ML concepts for interviews: data preparation, feature engineering, model selection, evaluation, bias-variance trade-off, interpretability, and deployment.

## Table of Contents

1. **Data Preparation & Feature Engineering** — cleaning, encoding, scaling, feature creation
2. **Model Selection & Training** — classification, regression, clustering, trade-offs
3. **Evaluation & Validation** — metrics, cross-validation, train/val/test splits
4. **Bias, Variance & Generalization** — overfitting, underfitting, regularization
5. **Interpretability & Responsible AI** — explainability, bias detection, fairness
6. **Deployment Awareness** — reproducibility, scalability, monitoring

---
## 1. Data Preparation & Feature Engineering

**Why it matters:** Real-world data is messy. Cleaning and feature engineering often matter more than model choice.

### 🧠 Beginner's Guide
Think of data preparation like cooking: you need to wash, chop, and season ingredients before you can cook. Models are the same — they can't handle raw data well.

- **Missing values:** Data often has holes (empty cells). You can fill them with averages (mean/median), carry forward the last known value, or use a model to predict the missing value. The way values go missing matters — completely at random (MCAR), related to other data (MAR), or related to the missing value itself (MNAR).
- **Categorical encoding:** Models work with numbers, not words like "Engineer" or "Sales." One-hot encoding creates a separate column for each category. Target encoding replaces categories with their average target value — powerful but can leak information.
- **Scaling:** Features on different scales (age 0-100 vs salary 0-1,000,000) confuse linear models. StandardScaler subtracts the mean and divides by standard deviation (z-score). MinMaxScaler squeezes values into [0,1]. Tree-based models don't care about scale.
- **Feature creation:** Raw data is rarely enough. Creating ratios (debt/income), date parts (day of week), or text counts helps models find patterns. Domain knowledge is your superpower here.
- **Outlier handling:** Extreme values can pull linear models off track. Z-score flags values more than 3 std deviations away. IQR uses quartiles. Tree models handle outliers naturally.
- **Imbalanced data:** If 99% of transactions are legitimate and 1% is fraud, a model that always says "legitimate" is 99% accurate but useless. Techniques: SMOTE creates synthetic minority samples; class weights penalize mistakes on the minority class more.

### Key Concepts

| Task | Techniques | Notes |
|---|---|---|
| **Missing values** | Mean/median/mode imputation, forward/backward fill, model-based imputation (KNN, MICE), drop rows/cols | Understand MCAR, MAR, MNAR mechanisms |
| **Categorical encoding** | One-hot, label/ordinal, target/mean encoding, frequency encoding, embeddings for high-cardinality | Watch for data leakage with target encoding |
| **Normalization / Scaling** | StandardScaler (z-score), MinMaxScaler (0–1), RobustScaler (IQR), PowerTransformer (skewed data) | Tree models don't need scaling; linear models & neural nets do |
| **Feature creation** | Polynomial features, binning, date/time decomposition, domain-specific ratios (e.g. P/E ratio), text TF-IDF / word vectors | Creativity + domain knowledge are key |
| **Outlier handling** | Z-score, IQR, winsorization, capping, isolation forest | Some models (tree-based) are robust to outliers; linear models are not |
| **Imbalanced data** | Resampling (SMOTE, ADASYN, random undersample), class weights, anomaly detection framing | Accuracy is misleading — use precision/recall/AUC |

### Code Examples

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

# Example: cleaning & preprocessing pipeline
np.random.seed(42)
df = pd.DataFrame({
    "age": [25, 30, np.nan, 35, 40, np.nan, 28],
    "salary": [50000, 60000, 55000, np.nan, 80000, 65000, 72000],
    "department": ["Eng", "Sales", "Eng", "Sales", "Mgmt", "Eng", "Mgmt"],
    "target": [0, 1, 0, 1, 1, 0, 1]
})

# 1. Handle missing values
imputer_num = SimpleImputer(strategy="median")
df[["age", "salary"]] = imputer_num.fit_transform(df[["age", "salary"]])

# 2. Encode categoricals
df = pd.get_dummies(df, columns=["department"], prefix="dept")

# 3. Scale features
scaler = StandardScaler()
df[["age", "salary"]] = scaler.fit_transform(df[["age", "salary"]])

print("Preprocessed DataFrame:")
print(df)


Preprocessed DataFrame:
        age    salary  target  dept_Eng  dept_Mgmt  dept_Sales
0 -1.350360 -1.440860       0      True      False       False
1 -0.251230 -0.373556       1     False      False        True
2 -0.251230 -0.907208       0      True      False       False
3  0.847900 -0.106730       1     False      False        True
4  1.947030  1.761051       1     False       True       False
5 -0.251230  0.160096       0      True      False       False
6 -0.690882  0.907208       1     False       True       False


### Discussion Questions — Data Preparation

- **"How do you handle missing data in a production pipeline?"** → Use a fitted imputer saved with the model. Never `fit` on production data — only `transform`. Consider whether missingness is informative (e.g. "no credit history" = risk signal). *Beginner tip: Think of `fit` as "learning" the median of training data, and `transform` as "applying" it. You don't want production data to influence the imputer's parameters.*
- **"What's the danger of target encoding?"** → Leaks information from the target into features. Always perform encoding inside cross-validation folds (or use holdout encoding). *Beginner tip: If you use the whole dataset's target average before splitting, the model gets a "sneak peek" at the target — like looking at answers before an exam.*
- **"When would you avoid one-hot encoding?"** → High-cardinality features (e.g. 10,000+ categories) explode dimensionality. Alternatives: frequency encoding, target encoding, or learned embeddings. *Beginner tip: One-hot 1000 categories = 1000 extra columns. That's too many! Frequency encoding replaces categories with how often they appear — just one column.*

---
## 2. Model Selection & Training

**Why it matters:** Different problems need different algorithms. Understanding trade-offs helps you pick the right tool.

### 🧠 Beginner's Guide
Imagine you're choosing a tool from a toolbox — you wouldn't use a hammer to screw in a nail. ML models are the same: each has strengths and weaknesses.

- **Classification** (predicting a category — spam/not-spam, fraud/legit): Start with **Logistic Regression** (fast, interpretable baseline). Upgrade to **Random Forest** (handles complex patterns, gives feature importance). If you need top performance, try **XGBoost/LightGBM** (gradient boosting — state-of-the-art for tabular data).
- **Regression** (predicting a number — stock price, temperature): **Linear Regression** is the simplest baseline. The same progression applies: Linear → Random Forest → XGBoost.
- **Clustering** (finding groups without labels — customer segments): **k-Means** is the go-to starting point (pick k using the elbow method). **DBSCAN** finds arbitrary shapes without needing to specify k.
- **Interpretability** matters in regulated industries (finance, healthcare). Linear models are fully transparent; tree ensembles are harder to explain; neural networks are black boxes.
- **Data size** matters: small data → simple models (linear/logistic/naive bayes). Large data → complex models (XGBoost, neural nets).
- **No free lunch:** No single algorithm is best for everything. Always try multiple models and cross-validate.

### Algorithm Comparison

| Model | Type | Interpretability | Strengths | Weaknesses |
|---|---|---|---|---|
| **Linear Regression** | Regression | ★★★★★ | Simple, fast, explainable | Assumes linearity, sensitive to outliers |
| **Logistic Regression** | Classification | ★★★★☆ | Well-calibrated probabilities, fast | Linear decision boundary |
| **Decision Tree** | Both | ★★★★★ | Non-linear, no scaling needed | Prone to overfitting, high variance |
| **Random Forest** | Both | ★★★☆☆ | Robust, handles non-linearity, feature importance | Slow with many trees, memory-heavy |
| **XGBoost / LightGBM** | Both | ★★☆☆☆ | State-of-the-art for tabular data, handles missing values | Many hyperparameters, can overfit |
| **SVM (RBF kernel)** | Both | ★☆☆☆☆ | Works well in high-dim spaces | Not scalable to large datasets |
| **k-NN** | Both | ★★★★☆ | Simple, non-parametric | Curse of dimensionality, slow prediction |
| **k-Means** | Clustering | ★★★★☆ | Fast, simple | Assumes spherical clusters, need to pick k |
| **DBSCAN** | Clustering | ★★★☆☆ | No k needed, finds arbitrary shapes | Struggles with varying density |

| **Neural Network** | Both | ★☆☆☆☆ | Extreme flexibility, SOTA for images/text/audio | Needs lots of data, hard to tune, black box |```

└─ Clustering → k-Means (baseline) → DBSCAN / HDBSCAN → GMM

### How to Choose├─ Regression → Linear Regression (baseline) → RF / XGBoost → NN

├─ Classification → Logistic Regression (baseline) → RF / XGBoost → NN

```Problem type?

In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.datasets import make_classification

# Quick comparison: Logistic Regression vs Random Forest
X, y = make_classification(n_samples=1000, n_features=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))
    print(f"{name:25s}  Accuracy: {acc:.3f}")


Logistic Regression        Accuracy: 0.830
Random Forest              Accuracy: 0.880


### Discussion Questions — Model Selection

- **"When would you pick a simple model over a complex one?"** → When you have limited data, need interpretability (regulated industries), or the simple model performs comparably (Occam's Razor). *Beginner tip: A linear regression with 5 features that runs in 1 second and is fully explainable often beats a neural net with 1M parameters that nobody can explain — especially when you have only 1,000 rows of data.*
- **"How do you know when a model is good enough?"** → Compare against a business baseline (not just random). A model that's 51% accurate but costs millions in bad decisions may not be "good enough." *Beginner tip: If the current system already operates at 85% accuracy, your model needs to beat that — not just beat random 50%. Always ask: "better than what?"*
- **"What's the no-free-lunch theorem in practice?"** → No single algorithm works best for all problems. Always try multiple models and validate. *Beginner tip: The algorithm that won the Kaggle competition for house prices might perform terribly on your spam detection problem. Try 3-5 different model families and cross-validate before committing.*

---
## 3. Evaluation & Validation

**Why it matters:** A model can look great on training data but fail in production. Proper evaluation is non-negotiable.

### 🧠 Beginner's Guide
Think of model evaluation like a student taking an exam. The training data is the textbook (memorising it), and the test is new questions they've never seen. A student who memorised answers without understanding will fail the test — that's **overfitting**.

---

### Classification Metrics — Deep Dive

**Accuracy** = $\frac{\text{correct predictions}}{\text{total predictions}}$

Simple and intuitive — but dangerous! If 99% of your transactions are legitimate and 1% is fraud, a model that always says "legitimate" achieves 99% accuracy but is completely useless. **Never use accuracy alone on imbalanced data.**

**The Confusion Matrix** is the foundation of all classification metrics:

```
                ┌─────────────────┬─────────────────┐
                │  Actually YES   │  Actually NO    │
├───────────────┼─────────────────┼─────────────────┤
│ Predicted YES │   True Positive │   False Positive│
│               │      (TP)       │      (FP)       │
├───────────────┼─────────────────┼─────────────────┤
│ Predicted NO  │   False Negative│   True Negative │
│               │      (FN)       │      (TN)       │
└───────────────┴─────────────────┴─────────────────┘
```

**Precision** = $\frac{TP}{TP + FP}$ — "Of everything I flagged as positive, how much was correct?"
- *Intuition:* When the model says "fraud," how often is it right? High precision = few false alarms.
- *Use when:* False positives are expensive (e.g. blocking legitimate transactions).

**Recall (Sensitivity)** = $\frac{TP}{TP + FN}$ — "Of all the actual positives, how many did I catch?"
- *Intuition:* What fraction of frauds did I actually detect? High recall = few missed positives.
- *Use when:* False negatives are expensive (e.g. missing a disease diagnosis).

**F1 Score** = $2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$ — The harmonic mean of precision and recall.

Why **harmonic mean** instead of regular average? Because it punishes extreme imbalance. If precision = 1.0 but recall = 0.0 (you make one perfect guess but miss everything else), the harmonic mean is 0 (correctly!), while the arithmetic mean would be 0.5 (misleadingly okay).

*Intuition:* F1 is your single-number score when you care about both false positives AND false negatives. It only gets high when BOTH precision and recall are high.

| Precision | Recall | Arithmetic Mean | F1 (Harmonic) |
|-----------|--------|----------------|---------------|
| 1.0 | 0.0 | 0.50 | **0.00** ✓ |
| 0.9 | 0.9 | 0.90 | **0.90** |
| 1.0 | 0.5 | 0.75 | **0.67** ✓ |
| 0.7 | 0.7 | 0.70 | **0.70** |

**AUC-ROC** = Area Under the Receiver Operating Characteristic curve. Plots True Positive Rate (Recall) vs False Positive Rate at all possible thresholds. AUC = 0.5 is random guessing; AUC = 1.0 is perfect.

**AUC-PR** = Area Under the Precision-Recall curve. Better than AUC-ROC for imbalanced datasets because it focuses on the positive class.

---

### Regression Metrics — Deep Dive

**MAE (Mean Absolute Error)** = $\frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i|$

*Intuition:* "On average, how far off are my predictions, in the original units?" If you're predicting house prices in dollars and MAE = $10,000, your typical prediction is $10k off.

- **Pros:** Easy to interpret, robust to outliers (each error counts equally)
- **Cons:** Doesn't tell you if you're consistently over or under; gradients aren't smooth for optimization

**MSE (Mean Squared Error)** = $\frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$

*Intuition:* Square the errors before averaging. This **heavily penalises large errors**. An error of 10 contributes 100 to MSE, while two errors of 5 each contribute only 50 total.

- **Pros:** Mathematically convenient (differentiable), penalises large mistakes heavily
- **Cons:** In squared units (hard to interpret), very sensitive to outliers

**RMSE (Root Mean Squared Error)** = $\sqrt{\frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2} = \sqrt{MSE}$

*Intuition:* Takes MSE and brings it back to the original units. If MSE = 100 (in $²), RMSE = $10. Now you can say "my typical error is about $10."

- **Pros:** Same units as target, penalises large errors
- **Cons:** Still affected by outliers (just less confusingly than MSE)

**MAE vs RMSE — Which one matters?**

| Scenario | MAE tells you | RMSE tells you |
|---|---|---|
| Mostly small errors, rare huge ones | ~5 | ~15 (huge errors dominate!) |
| Uniform errors | ~10 | ~10 (both agree) |
| Many medium errors | ~10 | ~11 |

*Key insight:* If MAE and RMSE are close, errors are uniform. If RMSE >> MAE, you have some massive errors dragging RMSE up.

**R² (Coefficient of Determination)** = $1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}$

*Intuition:* "What fraction of the variance in y can my model explain?"

- **R² = 1:** Perfect fit — the model explains all variance
- **R² = 0:** The model is no better than just predicting the mean $\bar{y}$
- **R² < 0:** The model is WORSE than predicting the mean (usually means something is badly wrong)

*Beginner tip:* R² answers "compared to guessing the average, how much better am I?" If R² = 0.75, you've explained 75% of the variance that the "average guess" couldn't.

**⚠️ R² can be misleading for non-linear models or when used outside the training range.** Adjusted R² adds a penalty for unnecessary features.

---

**Validation:** Never evaluate on the data you trained on (that's cheating!). Always hold out a test set. Better yet, use **k-fold cross-validation** — split data into k groups, train on k-1 groups, test on the held-out group, repeat k times. This gives a more reliable estimate.

### Metrics by Problem Type

| Problem | Metrics | Notes |
|---|---|---|
| **Binary Classification** | Accuracy, Precision, Recall, F1, AUC-ROC, AUC-PR, Log Loss | Use AUC-ROC for balanced; AUC-PR for imbalanced |
| **Multi-class** | Macro/micro/weighted F1, Cohen's Kappa, Confusion Matrix | Micro F1 = accuracy for multi-class |
| **Regression** | MSE, RMSE, MAE, R², Adjusted R², MAPE | RMSE penalises large errors; MAE is more robust |
| **Clustering** | Silhouette Score, Davies-Bouldin, Inertia (elbow), Adjusted Rand Index | Need ground truth for ARI; silhouette for unsupervised |

### Validation Strategies

| Strategy | Use Case | Pros | Cons |
|---|---|---|---|
| **Hold-out (70/30)** | Large datasets | Fast | High variance estimate |
| **k-Fold CV (k=5 or 10)** | General purpose | Low bias, low variance | 5-10x slower |
| **Stratified k-Fold** | Imbalanced classification | Preserves class proportions | Slightly more complex |
| **Time Series CV** | Temporal data | Prevents look-ahead bias | Cannot use future to predict past |
| **Leave-One-Out (LOO)** | Very small datasets (<100 samples) | Uses all data for training | Very expensive for large N |
| **Group k-Fold** | Grouped data (e.g. same patient) | Prevents group leakage | Requires group labels |

### Visualising Metrics — F1 Score, R², MAE, RMSE Explained with Graphs

The graphs below bring the metrics above to life — showing how F1 balances precision & recall, how MAE vs RMSE differ with outliers, and what R² values look like visually.

In [ ]:
# ── F1 Score, R², MAE, RMSE — Visual Explanations ──

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.linear_model import LinearRegression

plt.rcParams["figure.figsize"] = (14, 12)
plt.rcParams["font.size"] = 10

fig, axes = plt.subplots(3, 2)

# ═══════════════════════════════════════════════
# 1. F1 SCORE — Precision vs Recall Trade-off
# ═══════════════════════════════════════════════
ax = axes[0, 0]
recall_vals = np.linspace(0.1, 1, 50)
for prec in [0.3, 0.5, 0.7, 0.9, 1.0]:
    f1 = 2 * (prec * recall_vals) / (prec + recall_vals)
    ax.plot(recall_vals, f1, label=f"Precision={prec:.1f}")

ax.set_xlabel("Recall")
ax.set_ylabel("F1 Score")
ax.set_title("F1 Score = 2 × (P × R) / (P + R)\nHarmonic Mean of Precision & Recall", fontsize=11)
ax.legend(loc="lower right", fontsize=8)
ax.grid(True, alpha=0.3)
ax.annotate("F1 = 0 if either\nprecision or\nrecall is 0",
            xy=(0.1, 0.1), fontsize=8, color="red",
            bbox=dict(boxstyle="round,pad=0.3", fc="lightyellow", ec="red", alpha=0.8))

# ═══════════════════════════════════════════════
# 2. CONFUSION MATRIX — foundation for F1
# ═══════════════════════════════════════════════
ax = axes[0, 1]
ax.axis("off")
conf_data = [
    ["", "Actually YES", "Actually NO"],
    ["Predicted YES", "TP = 85", "FP = 15  ← Type I"],
    ["Predicted NO", "FN = 25  ← Type II", "TN = 875"],
]
table = ax.table(cellText=conf_data, loc="center", cellLoc="center", colWidths=[0.2, 0.35, 0.35])
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 2.5)
for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_facecolor("#404060")
        cell.set_text_props(color="white", fontweight="bold")
    elif col == 0:
        cell.set_facecolor("#e0e0e0")
        cell.set_text_props(fontweight="bold")

ax.set_title("Confusion Matrix (Example: 1000 predictions)\nPrecision=85/(85+15)=0.85  Recall=85/(85+25)=0.77  F1=0.81",
             fontsize=10, pad=10)

# ═══════════════════════════════════════════════
# 3. MAE vs RMSE — Comparison
# ═══════════════════════════════════════════════
ax = axes[1, 0]
np.random.seed(42)
n_pts = 50
x_plot = np.arange(n_pts)
true_vals = 10 + np.sin(np.linspace(0, 6, n_pts)) * 3
pred_a = true_vals + np.random.normal(0, 0.8, n_pts)  # small uniform errors
pred_b = true_vals.copy()
pred_b[:] += np.random.normal(0, 0.5, n_pts)
pred_b[25] = true_vals[25] + 12  # massive outlier

mae_a = mean_absolute_error(true_vals, pred_a)
rmse_a = np.sqrt(mean_squared_error(true_vals, pred_a))
mae_b = mean_absolute_error(true_vals, pred_b)
rmse_b = np.sqrt(mean_squared_error(true_vals, pred_b))

ax.plot(x_plot, true_vals, "k-", lw=2, alpha=0.8, label="True values")
ax.plot(x_plot, pred_b, "ro", ms=3, alpha=0.6, label="Predictions (with outlier)")
ax.plot(x_plot, pred_a, "bo", ms=3, alpha=0.4, label="Predictions (uniform)")
ax.annotate("Outlier! Error = 12",
            xy=(25, pred_b[25]), xytext=(30, pred_b[25]+2),
            fontsize=9, color="red", fontweight="bold",
            arrowprops=dict(arrowstyle="->", color="red"))

ax.set_title("MAE vs RMSE — Outlier Sensitivity\n"
             f"Uniform: MAE={mae_a:.2f}, RMSE={rmse_a:.2f} (close ≈ same)\n"
             f"With outlier: MAE={mae_b:.2f}, RMSE={rmse_b:.2f} (RMSE >> MAE!)",
             fontsize=10)
ax.legend(fontsize=8)
ax.set_xlabel("Sample index")
ax.set_ylabel("Value")
ax.grid(True, alpha=0.3)

# ═══════════════════════════════════════════════
# 4. R² — Visual intuition
# ═══════════════════════════════════════════════
ax = axes[1, 1]
np.random.seed(42)
X_r2 = np.linspace(0, 5, 30)
y_r2 = 2 + 1.5 * X_r2 + np.random.normal(0, 1.2, 30)
model_r2 = LinearRegression()
model_r2.fit(X_r2.reshape(-1, 1), y_r2)
y_pred_r2 = model_r2.predict(X_r2.reshape(-1, 1))

r2 = r2_score(y_r2, y_pred_r2)

ax.axhline(y=y_r2.mean(), color="gray", ls="--", lw=2, alpha=0.7,
           label=f'Mean baseline (ȳ = {y_r2.mean():.2f})')
ax.plot(X_r2, y_r2, "ko", ms=4, label="Actual data")
ax.plot(X_r2, y_pred_r2, "b-", lw=2, label=f"Model prediction (R² = {r2:.3f})")

idx = 10
ax.vlines(X_r2[idx], y_r2.mean(), y_pred_r2[idx], colors="green", lw=2, alpha=0.6,
          label=f"Explained (model - mean)")
ax.vlines(X_r2[idx], y_pred_r2[idx], y_r2[idx], colors="red", lw=2, alpha=0.6,
          label=f"Unexplained (actual - model)")

ax.set_title(f"R² = 1 − SSE_model / SSE_mean\n"
             f"R² = {r2:.3f} → Model explains {r2*100:.1f}% of variance",
             fontsize=10)
ax.legend(fontsize=7, loc="upper left")
ax.set_xlabel("Feature X")
ax.set_ylabel("Target y")
ax.grid(True, alpha=0.3)

# ═══════════════════════════════════════════════
# 5. MAE vs RMSE — Error distribution histogram
# ═══════════════════════════════════════════════
ax = axes[2, 0]
errors_a = pred_a - true_vals
errors_b = pred_b - true_vals

ax.hist(errors_a, bins=15, alpha=0.6, color="blue",
        label=f"Uniform errors\nMAE={mae_a:.2f}, RMSE={rmse_a:.2f}")
ax.hist(errors_b, bins=15, alpha=0.4, color="red",
        label=f"With outlier\nMAE={mae_b:.2f}, RMSE={rmse_b:.2f}")

ax.axvline(0, color="k", ls="--", lw=1)
ax.set_title("Error Distribution — MAE vs RMSE\n"
             "RMSE is pulled higher by the outlier (squared error dominates)",
             fontsize=10)
ax.legend(fontsize=8)
ax.set_xlabel("Prediction Error")
ax.set_ylabel("Frequency")
ax.grid(True, alpha=0.3)

# ═══════════════════════════════════════════════
# 6. R² Values — Good, Mediocre, Bad
# ═══════════════════════════════════════════════
ax = axes[2, 1]
x_r2demo = np.linspace(0, 5, 20)
np.random.seed(42)

r2_examples = [
    ("R² = 0.95  (Excellent)", 0.3),
    ("R² = 0.60  (Decent)", 1.5),
    ("R² = 0.10  (Poor)", 4.0),
    ("R² < 0  (Worse than mean!)", 8.0),
]

for label, noise_level in r2_examples:
    y_demo = 2 + 1.5 * x_r2demo + np.random.normal(0, noise_level, 20)
    pred_demo = 2 + 1.5 * x_r2demo
    r2_val = r2_score(y_demo, pred_demo)
    ax.plot(x_r2demo, y_demo, "o", ms=3, label=f"{label} (actual R²={r2_val:.2f})")

ax.plot(x_r2demo, 2 + 1.5 * x_r2demo, "k-", lw=2, alpha=0.7, label="True relationship")
ax.set_title("R² Across Different Noise Levels\n"
             "More noise → lower R² (same underlying relationship!)",
             fontsize=10)
ax.legend(fontsize=7, loc="upper left")
ax.set_xlabel("Feature X")
ax.set_ylabel("Target y")
ax.grid(True, alpha=0.3)

plt.tight_layout(pad=2)
plt.show()


In [3]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix

# Stratified k-fold cross-validation
model = RandomForestClassifier(n_estimators=50, random_state=42)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X, y, cv=cv, scoring="f1")

print(f"F1 scores per fold: {scores}")
print(f"Mean F1: {scores.mean():.3f} (±{scores.std():.3f})")

# Full train/test with detailed metrics
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


F1 scores per fold: [0.87234043 0.91       0.88780488 0.94300518 0.89361702]
Mean F1: 0.901 (±0.024)

Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.91      0.88        89
           1       0.92      0.86      0.89       111

    accuracy                           0.89       200
   macro avg       0.88      0.89      0.88       200
weighted avg       0.89      0.89      0.89       200



### Discussion Questions — Evaluation

- **"Why use AUC-ROC vs AUC-PR?"** → PR curves focus on the positive class — use AUC-PR for imbalanced datasets. AUC-ROC can look optimistic when negatives dominate. *Beginner tip: Imagine 99 normal transactions and 1 fraud. A model that always says "normal" gets AUC-ROC = 0.5 (random) — but AUC-PR would correctly show it's useless because it never catches the fraud.*
- **"How do you know if two models are significantly different?"** → Use a paired statistical test (e.g. McNemar's test for classifiers, paired t-test across CV folds). *Beginner tip: If model A gets 87.1% and model B gets 87.3%, is that real or just random? Statistical tests tell you whether the difference is bigger than expected from noise.*
- **"What's the danger of data leakage in CV?"** → If preprocessing (scaling, encoding) is fit on the full dataset before splitting, information from the test fold leaks into training. Always fit preprocessing inside the CV loop. *Beginner tip: If you calculate the global mean of a feature using all data (including test), then the test data has "seen" the training data's mean information. Your test score will be overly optimistic — the model will look better than it actually is.*

---
## 4. Bias, Variance & Generalization

**Why it matters:** The central tension in ML — fitting the data well without fitting the noise.

### 🧠 Beginner's Guide
Imagine you're learning to play darts.

- **High bias (underfitting):** You always throw to the same spot, but it's far from the bullseye. You're not learning enough from the data. → *Fix: use a more complex model, add more features.*
- **High variance (overfitting):** You hit the bullseye a few times in practice, but on a new day you miss completely. You've memorised the noise instead of the pattern. → *Fix: get more data, simplify the model, use regularization.*
- **The sweet spot:** You consistently hit near the bullseye even on new days. The model generalises.

**Error = Bias² + Variance + Irreducible Error**

- **Bias:** Error from wrong assumptions (e.g. assuming a linear relationship when it's actually a curve).
- **Variance:** Error from sensitivity to small fluctuations in the training data.
- **Irreducible Error:** Noise inherent in the data — you can't reduce this.

**Regularization** is like adding rules to prevent the model from fitting the noise. L1 (Lasso) can zero out unimportant features entirely. L2 (Ridge) shrinks all weights toward zero but keeps them. Dropout randomly turns off neurons during neural net training so the network learns redundant patterns.

### The Bias-Variance Trade-off

```
                    Underfitting ← → Overfitting
                    High Bias       Low Bias
                    Low Variance    High Variance
                        
Error = Bias² + Variance + Irreducible Error
```

| Symptom | Likely Cause | Fix |
|---|---|---|
| High training error, high test error | **Underfitting (high bias)** | More complex model, more features, fewer constraints |
| Low training error, high test error | **Overfitting (high variance)** | Regularization, more data, simpler model, early stopping |
| Low training error, low test error | **Just right** | — |

### Regularization Techniques

| Technique | Applies To | How It Works |
|---|---|---|
| **L1 (Lasso)** | Linear models | Adds $\lambda \sum \lvert w_i \rvert$ — drives weights to zero (feature selection) |
| **L2 (Ridge)** | Linear models | Adds $\lambda \sum w_i^2$ — shrinks weights but keeps all features |
| **ElasticNet** | Linear models | Combines L1 + L2 |
| **Dropout** | Neural networks | Randomly drops neurons during training |
| **Early stopping** | Iterative models | Stop training when validation loss stops improving |
| **Pruning** | Decision trees | Remove branches with little predictive power |
| **Data augmentation** | Images/text | Create synthetic training examples |
| **Ensembling** | All models | Bagging (RF) reduces variance; Boosting (XGB) reduces bias |

| **Dropout** | Neural networks | Randomly drops neurons during training |

| Technique | Applies To | How It Works || **ElasticNet** | Linear models | Combines L1 + L2 |

|---|---|---|| **L2 (Ridge)** | Linear models | Adds $\lambda \sum w_i^2$ — shrinks weights but keeps all features |
| **L1 (Lasso)** | Linear models | Adds $\lambda \sum \lvert w_i \rvert$ — drives weights to zero (feature selection) |

In [4]:
# Demonstrate overfitting visually with polynomial regression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Ridge, LinearRegression

np.random.seed(42)
X_small = np.linspace(0, 1, 10).reshape(-1, 1)
y_small = np.sin(2 * np.pi * X_small).ravel() + np.random.normal(0, 0.2, 10)

# High-degree poly without regularization → overfits
overfit = make_pipeline(PolynomialFeatures(15), LinearRegression())
overfit.fit(X_small, y_small)
train_err = ((overfit.predict(X_small) - y_small) ** 2).mean()

# Same degree with Ridge (L2) regularization → controlled
regularized = make_pipeline(PolynomialFeatures(15), Ridge(alpha=1.0))
regularized.fit(X_small, y_small)
reg_train_err = ((regularized.predict(X_small) - y_small) ** 2).mean()

print(f"Overfit model train MSE:    {train_err:.4f}")
print(f"Regularized model train MSE: {reg_train_err:.4f}")
print("(Overfit model would have much higher test error on unseen data)")


Overfit model train MSE:    0.0000
Regularized model train MSE: 0.2222
(Overfit model would have much higher test error on unseen data)


### Discussion Questions — Bias & Variance

- **"How do you diagnose overfitting without a test set?"** → Training loss << validation loss, weights growing very large, high variance across CV folds. *Beginner tip: If training accuracy is 99.9% but validation accuracy is 70%, that's the classic overfitting signal — the model memorised the training data instead of learning general patterns.*
- **"What happens to bias and variance as tree depth increases?"** → Bias decreases, variance increases. At depth = 1, high bias (stump). At max depth, near-zero bias but very high variance. *Beginner tip: A depth-1 tree is just one rule ("if age > 30 predict A") — too simple. A depth-100 tree memorises every single training example — too complex. The right depth is somewhere in between.*
- **"Why does bagging reduce variance?"** → Averaging N independent models reduces variance by factor of ~1/N (assuming uncorrelated errors). Random Forest decorrelates trees via feature sampling. *Beginner tip: One expert might be wrong, but 100 experts voting together are much more reliable — especially if each expert focuses on different features.*
- **"How does XGBoost handle the bias-variance trade-off?"** → Uses a regularised objective ($\gamma$ for tree complexity, $\lambda$ for leaf weights, $\alpha$ for L1), shrinkage (learning rate), and early stopping. *Beginner tip: XGBoost builds trees one at a time, each new tree correcting the previous tree's mistakes. Regularization prevents this process from going too wild — like a teacher who says "fix your mistakes, but don't overthink it."*

---
## 5. Interpretability & Responsible AI

**Why it matters:** Regulations (EU AI Act, GDPR), high-stakes decisions (credit, healthcare), and debugging require understanding *why* a model makes predictions.

### 🧠 Beginner's Guide
Imagine a bank denies your loan application. A black-box model says "denied" but can't explain why. That's not just frustrating — it's illegal in many jurisdictions. Interpretability answers "why did the model do that?"

- **Global interpretability:** Understanding the whole model at once ("Age and income are the top 2 most important features"). Methods: Feature importance, Partial Dependence Plots.
- **Local interpretability:** Understanding a single prediction ("Your loan was denied because income is low AND debt ratio is high"). Methods: SHAP, LIME, Counterfactuals.

**SHAP** is the gold standard — it uses game theory to fairly distribute the prediction among features. Each feature gets a "SHAP value" showing how much it pushed the prediction up or down from the baseline. SHAP values are consistent and additive (they sum to the prediction).

**Fairness:** Models can be biased even if we don't explicitly include protected attributes (race, gender). The model might learn proxies (zip code → race). Three common fairness definitions:
- **Demographic parity:** Equal acceptance rates across groups.
- **Equal opportunity:** Equal true positive rates across groups (catch the same proportion of qualified candidates).
- **Equalised odds:** Both TPR and FPR are equal across groups (hardest to satisfy).

**The catch:** You can't satisfy all fairness definitions at once — you must choose based on the business/legal context.

### Interpretability Methods

| Method | Scope | How It Works |
|---|---|---|
| **Feature importance (impurity)** | Global | Tree-based: total reduction in criterion (Gini/entropy) across splits |
| **Permutation importance** | Global | Shuffle a feature and measure drop in performance — model-agnostic |
| **Partial Dependence Plot (PDP)** | Global | Vary one feature while marginalising others — shows average relationship |
| **SHAP** | Local + Global | Game-theoretic: each feature gets a "contribution" to the prediction. Consistent and additive |
| **LIME** | Local | Fit a simple surrogate model around a single prediction |
| **Integrated Gradients** | Local | For neural networks: integrate gradients along path from baseline to input |
| **Grad-CAM** | Local (vision) | Heatmap of what regions a CNN "looked at" for its prediction |
| **Counterfactual explanations** | Local | "What's the smallest change to flip the prediction?" |

### Bias Detection & Fairness

3. **Post-processing:** Adjust decision thresholds per group to achieve parity.

| Concept | Definition |2. **In-processing:** Add fairness constraints to the loss function (e.g. adversarial debiasing).

|---|---|1. **Pre-processing:** Reweighing, relabelling, data augmentation to remove bias from training data.

| **Demographic parity** | Prediction rates are equal across groups |

| **Equal opportunity** | True positive rates are equal across groups |### Fairness Mitigation

| **Equalised odds** | Both TPR and FPR are equal across groups |

| **Disparate impact** | Ratio of positive outcome rates between groups (80% rule) |**Sources of bias:** Sampling bias, label bias, measurement bias, aggregation bias, historical bias, confirmation bias in feature selection.


In [5]:
# SHAP example (conceptual — requires shap library)
# import shap
# explainer = shap.TreeExplainer(model)
# shap_values = explainer.shap_values(X_test)
# shap.summary_plot(shap_values, X_test)

# Permutation importance — built into sklearn
from sklearn.inspection import permutation_importance

result = permutation_importance(
    model, X_test, y_test, n_repeats=5, random_state=42, scoring="f1"
)

for i in range(len(result.importances_mean)):
    print(f"Feature {i}: importance = {result.importances_mean[i]:.4f} "
          f"(±{result.importances_std[i]:.4f})")


Feature 0: importance = 0.0041 (±0.0033)
Feature 1: importance = -0.0010 (±0.0020)
Feature 2: importance = 0.0241 (±0.0044)
Feature 3: importance = -0.0004 (±0.0035)
Feature 4: importance = 0.0081 (±0.0030)
Feature 5: importance = 0.0037 (±0.0035)
Feature 6: importance = 0.2916 (±0.0353)
Feature 7: importance = 0.0025 (±0.0022)
Feature 8: importance = 0.0539 (±0.0130)
Feature 9: importance = -0.0029 (±0.0046)


### Discussion Questions — Interpretability & Fairness

- **"How do you explain an XGBoost model to a non-technical stakeholder?"** → Use SHAP summary plots (which features push predictions higher/lower), and highlight a few concrete examples with waterfall plots. *Beginner tip: Don't show them the math! Show a simple bar chart: "Income contributed +0.3 to the score, age contributed +0.1, debt ratio contributed -0.5." They understand "this helped / this hurt."*
- **"What's the problem with feature importance from tree models?"** → High-cardinality features (e.g. 1000 categories) get inflated importance due to more split points. Permutation importance is more reliable. *Beginner tip: A feature with 1000 categories gets 999 chances to split on — it's like giving someone 999 lottery tickets vs someone else's 1 ticket. Permutation importance shuffles the feature and checks if performance drops, which is fairer.*
- **"How would you audit a model for fairness?"** → Split data by protected attributes (race, gender), compute metrics (TPR, FPR, precision) per group, check for statistically significant differences. Use tools like Aequitas or Fairlearn. *Beginner tip: If your loan model approves 80% of men but only 50% of equally-qualified women, that's a red flag. The first step is measuring these numbers honestly.*
- **"Can a model be fair by one definition and unfair by another?"** → Yes — demographic parity and equal opportunity can conflict. You must choose definitions that align with the business/legal context. *Beginner tip: Imagine a model that picks the best job candidates. Equal opportunity means qualified candidates have equal chance regardless of group. Demographic parity means the acceptance rate is the same. If one group has fewer qualified candidates, these definitions conflict — you can't satisfy both.*

---
## 6. Deployment Awareness

**Why it matters:** A model in a notebook has zero business value until it's deployed and monitored.

### 🧠 Beginner's Guide
Building a model in a Jupyter notebook is like baking a cake in a lab — the real challenge is mass-producing it consistently in a factory. Deployment is about getting that model to production reliably and keeping it running.

- **Reproducibility:** Six months later, can you recreate the same model? Pin ALL dependencies (Python packages, data versions, random seeds). Use Docker to containerise everything so "it works on my machine" isn't an excuse.
- **Serving:** How does the model make predictions?
  - *Batch:* Run predictions on millions of rows every night (cheap, but stale).
  - *Online:* API endpoint returns a prediction in milliseconds (fresh, but expensive).
  - *Streaming:* Process data as it arrives (Kafka + lightweight model).
- **Monitoring:** Models degrade over time because the world changes. Two types of drift:
  - *Data drift:* The input data changed (customers got older, new products launched). Monitor feature distributions with statistical tests (KS test, Population Stability Index).
  - *Concept drift:* The relationship between inputs and outputs changed (what "fraud" looks like evolves). Monitor prediction distributions and actuals (when labels arrive).
- **Retraining:** Once drift is detected, retrain the model. Options: scheduled (weekly), triggered (when metric drops below threshold), or continuous (online learning).
- **MLOps:** Apply DevOps practices to ML — version control for data and models, CI/CD pipelines that train and validate models before deployment, A/B testing challenger vs champion models.

**Golden rule:** If you can't monitor it, don't deploy it.

### The ML Lifecycle (Beyond Training)

```
Data → Feature Engineering → Training → Validation → Deployment → Monitoring → Retraining
                                                                          ↻
```

### Key Deployment Considerations

| Area | Details |
|---|---|
| **Reproducibility** | Pin dependencies (`requirements.txt` / `conda.yml`), version data (DVC, S3 with hashes), version models (MLflow Model Registry), seed everything |
| **Serving** | Batch (scheduled predictions), Online (REST/gRPC API), Streaming (Kafka + lightweight model) |
| **Latency & Throughput** | Prune models (ONNX, TensorRT), quantize (FP16/INT8), use smaller architectures (DistilBERT vs BERT), caching for repeated queries |
| **Containerisation** | Docker + orchestrator (Kubernetes, ECS). Separate build from run stages. Keep images small |
| **CI/CD for ML (MLOps)** | Test data quality, train in CI, evaluate against baseline, promote to staging, shadow-deploy, then production rollout |
| **Monitoring** | **Data drift** (feature distributions change), **Concept drift** (P(y\|x) changes), **Prediction drift** (output distribution shifts). Tools: Evidently, WhyLabs, NannyML |
| **Retraining strategies** | Scheduled (weekly), performance-threshold-triggered, or continuous (online learning). Monitor for staleness |
| **A/B Testing** | Deploy challenger vs champion, measure business metrics (not just model metrics). Ramp up traffic gradually |
| **Rollback** | Keep previous model versions available. Automate rollback on metric degradation (e.g. AUC drops > 0.02) |
| **Explainability in prod** | Log SHAP values per prediction for audit trails. Store input features + prediction + explanation together |

In [6]:
# Conceptual deployment pipeline — pseudo-code for an MLflow tracking example

# import mlflow
# mlflow.set_experiment("churn_prediction")
#
# with mlflow.start_run():
#     model = RandomForestClassifier(n_estimators=100)
#     model.fit(X_train, y_train)
#     mlflow.log_param("n_estimators", 100)
#     mlflow.log_metric("val_f1", f1_score(y_val, model.predict(X_val)))
#     mlflow.sklearn.log_model(model, "model")
#     # Later: mlflow.register_model("runs:/<run_id>/model", "churn_model_v2")

# Note on monitoring: track feature distributions with Evidently
# from evidently import ColumnMapping
# from evidently.report import Report
# from evidently.metric_preset import DataDriftPreset
#
# report = Report(metrics=[DataDriftPreset()])
# report.run(reference_data=df_train, current_data=df_production)
# report.save_html("drift_report.html")

print("MLflow & Evidently concepts shown above (commented — requires library install)")


MLflow & Evidently concepts shown above (commented — requires library install)


### Discussion Questions — Deployment

- **"What's the difference between data drift and concept drift?"** → *Data drift*: the input distribution P(X) changes (e.g. users got older). *Concept drift*: the relationship P(y|X) changes (e.g. what "fraud" looks like evolves). Both require retraining, but concept drift may need new labels. *Beginner tip: Data drift = the world changed. Concept drift = the rules changed. Example: During COVID, spending patterns changed (data drift) AND what counts as "risky spending" changed (concept drift).*
- **"How would you detect model degradation in production?"** → Track prediction distribution, feature distributions, and actuals (when they arrive with a delay). Use statistical tests (KS test, PSI) for drift. Set alert thresholds with appropriate lookback windows. *Beginner tip: Set up dashboards that track "average prediction per day" and "average age of users per day." If average prediction suddenly goes from 0.3 to 0.7, something changed — investigate!*
- **"How do you handle delayed ground truth?"** → Use proxy metrics (e.g. CTR for recommendations) while waiting for actual labels. Consider survival analysis or partial labels. *Beginner tip: If you only know if a loan defaulted after 12 months, you can't wait that long to evaluate. Use early signals: first missed payment, dropped insurance, etc.*
- **"When should you NOT deploy an ML model?"** → When the cost of errors outweighs the benefit, when there's no clear business metric improvement, when the model is unexplainable in a regulated context, or when simpler heuristics are sufficient. *Beginner tip: Sometimes a simple rule ("if > 3 late payments, flag as risky") beats a complex model because it's easy to debug, explain to regulators, and fix when broken. Don't use ML for the sake of using ML.*